# 05 — Text & Audio Feature Generators (Video)
Defines and tests two feature extraction pipelines that will be used
in the trimodal fusion notebook:

- **Text:** Structured sentence templates → DistilBERT → 768-d embedding
- **Audio:** Same sentence → gTTS (speech synthesis) → MFCC features → 120-d vector

> This notebook is a **helper / test notebook**.  
> Both generators are imported as utility functions in `06_video_trimodal_fusion.ipynb`.

## 1. Install dependencies

In [ ]:
!pip install gtts librosa transformers -q

## 2. Imports

In [ ]:
import os, random, tempfile
import numpy as np
import torch
from gtts import gTTS
import librosa
from transformers import DistilBertTokenizer, DistilBertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 3. Class names

In [ ]:
CLASS_NAMES = [
    'Basketball', 'Biking', 'Bowling', 'CliffDiving',
    'GolfSwing', 'HorseRiding', 'Skiing', 'Surfing',
    'TennisSwing', 'SkateBoarding'
]

## 4. Text generator
Structured sentence templates + class-specific word banks.
Stop words are inserted at random positions to simulate noisy, natural-language captions.
Noise is intentional: it prevents the text branch from trivially solving the task on its own,
forcing the model to fuse all three modalities.

In [ ]:
# ── Stop words (inserted randomly to simulate natural speech) ─────
STOP_WORDS = [
    'the', 'a', 'an', 'this', 'that', 'is', 'are', 'was',
    'in', 'on', 'at', 'with', 'and', 'of', 'for', 'to',
    'showing', 'featuring', 'captured', 'seen', 'during'
]

# ── Sentence templates ────────────────────────────────────────────
TEMPLATES = [
    'this is {adj1} footage of {cls} {action}',
    'the video is showing {adj1} {cls} {action} in {place}',
    'captured footage featuring {cls} {action} with {adj2} conditions',
    'this {adj1} video is of {cls} {action} during {time}',
    'footage showing {adj1} and {adj2} {cls} {action}',
]

# ── Per-class word banks ──────────────────────────────────────────
CLASS_SPEECH_WORDS = {
    'Basketball': {
        'adj1'  : ['professional', 'competitive', 'indoor', 'live'],
        'adj2'  : ['intense', 'exciting', 'fast', 'dynamic'],
        'action': ['players dribbling and shooting', 'match in progress', 'game underway'],
        'place' : ['indoor court', 'basketball arena', 'sports hall'],
        'time'  : ['the match', 'halftime', 'championship game']
    },
    'Biking': {
        'adj1'  : ['outdoor', 'fast', 'competitive', 'scenic'],
        'adj2'  : ['challenging', 'rough', 'open', 'clear'],
        'action': ['rider cycling on road', 'person riding bicycle', 'cyclist in motion'],
        'place' : ['open road', 'cycling trail', 'outdoor track'],
        'time'  : ['the race', 'morning ride', 'training session']
    },
    'Bowling': {
        'adj1'  : ['indoor', 'competitive', 'professional', 'casual'],
        'adj2'  : ['precise', 'focused', 'clear', 'controlled'],
        'action': ['player throwing ball down lane', 'bowler aiming at pins', 'strike in progress'],
        'place' : ['bowling alley', 'indoor lane', 'sports center'],
        'time'  : ['the game', 'tournament', 'practice session']
    },
    'CliffDiving': {
        'adj1'  : ['extreme', 'breathtaking', 'outdoor', 'dangerous'],
        'adj2'  : ['high', 'rocky', 'steep', 'dramatic'],
        'action': ['athlete jumping from cliff', 'diver leaping into water', 'freefall in progress'],
        'place' : ['rocky cliff', 'ocean shore', 'natural diving spot'],
        'time'  : ['the jump', 'competition', 'training']
    },
    'GolfSwing': {
        'adj1'  : ['professional', 'outdoor', 'precise', 'calm'],
        'adj2'  : ['focused', 'clean', 'open', 'green'],
        'action': ['golfer swinging club', 'player hitting ball', 'golf shot in progress'],
        'place' : ['golf course', 'open fairway', 'green'],
        'time'  : ['the round', 'tournament', 'practice']
    },
    'HorseRiding': {
        'adj1'  : ['outdoor', 'graceful', 'competitive', 'rural'],
        'adj2'  : ['open', 'natural', 'calm', 'scenic'],
        'action': ['rider on horseback', 'person galloping on horse', 'equestrian in motion'],
        'place' : ['open field', 'equestrian track', 'countryside'],
        'time'  : ['the race', 'training', 'competition']
    },
    'Skiing': {
        'adj1'  : ['winter', 'fast', 'outdoor', 'snowy'],
        'adj2'  : ['steep', 'cold', 'icy', 'downhill'],
        'action': ['skier going down slope', 'person skiing at speed', 'downhill skiing in progress'],
        'place' : ['snowy mountain', 'ski slope', 'winter resort'],
        'time'  : ['the run', 'competition', 'training session']
    },
    'Surfing': {
        'adj1'  : ['outdoor', 'exciting', 'coastal', 'extreme'],
        'adj2'  : ['large', 'powerful', 'ocean', 'clear'],
        'action': ['surfer riding wave', 'person on surfboard', 'wave surfing in progress'],
        'place' : ['ocean shore', 'coastal waters', 'beach'],
        'time'  : ['the surf', 'competition', 'morning session']
    },
    'TennisSwing': {
        'adj1'  : ['competitive', 'outdoor', 'professional', 'fast'],
        'adj2'  : ['precise', 'powerful', 'focused', 'clean'],
        'action': ['player swinging racket', 'tennis shot in progress', 'serve and volley'],
        'place' : ['tennis court', 'outdoor court', 'sports arena'],
        'time'  : ['the match', 'tournament', 'practice session']
    },
    'SkateBoarding': {
        'adj1'  : ['outdoor', 'extreme', 'urban', 'fast'],
        'adj2'  : ['technical', 'smooth', 'urban', 'open'],
        'action': ['skater performing tricks', 'person riding skateboard', 'skateboard in motion'],
        'place' : ['skate park', 'urban area', 'outdoor ramp'],
        'time'  : ['the session', 'competition', 'practice']
    },
}


def generate_speech_text(cls_name):
    """Generate a noisy, natural-language caption for a given action class."""
    words    = CLASS_SPEECH_WORDS[cls_name]
    template = random.choice(TEMPLATES)
    sentence = template.format(
        cls    = cls_name.lower(),
        adj1   = random.choice(words['adj1']),
        adj2   = random.choice(words['adj2']),
        action = random.choice(words['action']),
        place  = random.choice(words['place']),
        time   = random.choice(words['time'])
    )
    # Insert 2-4 stop words at random positions
    word_list = sentence.split()
    for _ in range(random.randint(2, 4)):
        pos = random.randint(0, len(word_list))
        word_list.insert(pos, random.choice(STOP_WORDS))
    return ' '.join(word_list)


# Sanity check
print('Sample speech texts:')
for cls in CLASS_NAMES[:3]:
    print(f'\n{cls}:')
    print(f'  {generate_speech_text(cls)}')
    print(f'  {generate_speech_text(cls)}')

## 5. Text feature extractor (DistilBERT)
Converts text → [CLS] token embedding → 768-d vector.

In [ ]:
tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)

for p in bert_model.parameters():
    p.requires_grad = False
bert_model.eval()


@torch.no_grad()
def get_text_features(text):
    """Text string → (1, 768) tensor (CLS token embedding)."""
    tokens = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=64          # longer than image captions — sentences are richer
    ).to(device)
    out = bert_model(**tokens)
    return out.last_hidden_state[:, 0, :]   # (1, 768)


# Test
sample_text = generate_speech_text('Biking')
feat = get_text_features(sample_text)
print(f'Text feat shape: {feat.shape}  (expected: torch.Size([1, 768]))')
print('DistilBERT ready!')

## 6. Audio feature extractor (gTTS → MFCC)
Pipeline: text → gTTS MP3 → librosa → MFCC → aggregate (mean/std/max) → 120-d vector.

Why MFCC?
- Captures timbral texture (how words *sound*, not just what they mean)
- Compact: 40 coefficients × 3 stats = 120 floats
- Computationally cheap compared to raw audio waveforms

In [ ]:
def text_to_mfcc(text, n_mfcc=40, max_len=128):
    """
    Convert text → speech (gTTS) → MFCC features.
    Returns FloatTensor of shape (n_mfcc * 3,) = (120,).
    Falls back to zero vector on error.
    """
    try:
        # Step 1: Text → MP3 (using gTTS)
        tts = gTTS(text=text, lang='en', slow=False)
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
            tmp_path = f.name
        tts.save(tmp_path)

        # Step 2: Load audio waveform
        audio, sr = librosa.load(tmp_path, sr=22050)
        os.unlink(tmp_path)   # Clean up temp file immediately

        # Step 3: Extract MFCC → shape (n_mfcc, time)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

        # Pad or truncate to fixed time length
        if mfcc.shape[1] < max_len:
            mfcc = np.pad(mfcc, ((0, 0), (0, max_len - mfcc.shape[1])))
        else:
            mfcc = mfcc[:, :max_len]

        # Step 4: Aggregate across time → fixed-size feature vector
        feat = np.concatenate([
            np.mean(mfcc, axis=1),   # (40,)
            np.std( mfcc, axis=1),   # (40,)
            np.max( mfcc, axis=1),   # (40,)
        ])                           # → (120,)

        return torch.FloatTensor(feat)

    except Exception as e:
        print(f'[text_to_mfcc] Error: {e} — returning zeros')
        return torch.zeros(n_mfcc * 3)


# Test
print('Testing audio feature extraction...')
test_text = generate_speech_text('Skiing')
print(f'Input text: "{test_text}"')
audio_feat = text_to_mfcc(test_text)
print(f'Audio feat shape : {audio_feat.shape}  (expected: torch.Size([120]))')
print(f'Feature range    : [{audio_feat.min():.2f}, {audio_feat.max():.2f}]')
print('Audio pipeline ready!')

## 7. Full pipeline test — text + audio for all 10 classes

In [ ]:
print('Running full pipeline test for all classes:\n')
print(f'{"Class":15} {"Text feat":12} {"Audio feat":12} {"Sample text"}')
print('-' * 80)
for cls in CLASS_NAMES:
    text   = generate_speech_text(cls)
    t_feat = get_text_features(text)
    a_feat = text_to_mfcc(text)
    print(f'{cls:15} {str(tuple(t_feat.shape)):12} {str(tuple(a_feat.shape)):12} "{text[:40]}..."')

print('\nAll pipelines working correctly!')